# LipPY KenLM Domain-Matched (lippy-test 43K transcript)

**Sorun:** Mevcut KenLM Wikipedia (50K makale, formal Türkçe) ile eğitilmiş. Bizim training data konuşma dili (YouTube). Domain mismatch → real-world test'te KenLM yanlış kelime seçimi yapıyor.

**Çözüm:** lippy-test 43K transcript (training+val) ile yeni KenLM eğit. Domain-matched LM, konuşma dili kelimeleri yakalar.

## Cikti
- `lm_konusma.klm` — domain-matched KenLM (4-gram)
- `unigrams_konusma.txt` — filtered unigrams (len≥2, freq≥3)
- `corpus_konusma.txt` — birleştirilmiş train+val transcript
- Dataset: `brokyewing/lippy-kenlm-konusma`

## Karsilastirma (val 6461 üstünde, sonra)
- Wikipedia KenLM: WER 66.7% (training distribution iyi, real-world kötü)
- Konusma KenLM: WER ?% (real-world generalization daha iyi olmali)

## Sure
- Corpus build: 1 dk
- KenLM (lmplz) eğitim: 5-10 dk
- Binary build: 1 dk
- Test + upload: 5 dk
- **Toplam: ~20 dk** (CPU yeterli)

## Settings
- Accelerator: NONE (CPU)
- Internet: ON
- Inputs: `brokyewing/lippy-test`

## 1) Install KenLM (binary tools)

In [ ]:
import os, sys, glob, subprocess, shutil, json
from collections import Counter
from pathlib import Path

# KenLM binary tools (lmplz, build_binary) — apt'tan veya kaynaktan
print('Installing KenLM tools...')
!apt-get install -y libboost-all-dev cmake build-essential 2>&1 | tail -5

# kenlm clone + build (binary tools icin)
if not os.path.exists('/kaggle/working/kenlm'):
    !git clone -q https://github.com/kpu/kenlm.git /kaggle/working/kenlm
    !mkdir -p /kaggle/working/kenlm/build
    !cd /kaggle/working/kenlm/build && cmake .. > /dev/null 2>&1 && make -j4 2>&1 | tail -5
    print('KenLM tools build done')
else:
    print('KenLM tools already built')

LMPLZ = '/kaggle/working/kenlm/build/bin/lmplz'
BUILD_BINARY = '/kaggle/working/kenlm/build/bin/build_binary'

assert os.path.exists(LMPLZ), f'lmplz bulunamadi: {LMPLZ}'
assert os.path.exists(BUILD_BINARY), f'build_binary bulunamadi: {BUILD_BINARY}'
print(f'lmplz: {LMPLZ}')
print(f'build_binary: {BUILD_BINARY}')

# Python kenlm (sonradan test icin)
!pip install -q https://github.com/kpu/kenlm/archive/master.zip
import kenlm
print(f'kenlm Python: OK')

## 2) Corpus build — lippy-test transcript birleştir

In [ ]:
# Texts klasoru
text_cands = glob.glob('/kaggle/input/**/texts', recursive=True)
TEXT_DIR = text_cands[0] if text_cands else None
assert TEXT_DIR, 'texts/ bulunamadi'
print(f'Texts: {TEXT_DIR}')

txt_files = sorted(glob.glob(f'{TEXT_DIR}/*.txt'))
print(f'Toplam txt: {len(txt_files)}')

# Tum transcripti birlestir
WORK_DIR = '/kaggle/working'
CORPUS_PATH = f'{WORK_DIR}/corpus_konusma.txt'

import time
t0 = time.time()
lines = []
for fp in txt_files:
    try:
        with open(fp, 'r', encoding='utf-8') as f:
            txt = f.read().strip().lower()
        if txt:
            lines.append(txt)
    except Exception as e:
        print(f'Skip {fp}: {e}')

with open(CORPUS_PATH, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines) + '\n')

size_mb = os.path.getsize(CORPUS_PATH) / 1e6
print(f'\nCorpus: {len(lines)} satir, {size_mb:.1f} MB, {int(time.time()-t0)}s')

all_text = ' '.join(lines)
n_words = len(all_text.split())
n_chars = len(all_text)
print(f'Kelime: {n_words:,}, Karakter: {n_chars:,}')
print(f'Sample (ilk 3 cümle):')
for l in lines[:3]:
    print(f'  {l}')

## 3) KenLM 4-gram eğitim (lmplz)

In [ ]:
ARPA_PATH = f'{WORK_DIR}/lm_konusma.arpa'
KLM_PATH = f'{WORK_DIR}/lm_konusma.klm'
LMPLZ_LOG = f'{WORK_DIR}/lmplz.log'

ORDER = 4

print(f'KenLM lmplz egitimi (order={ORDER})...')
t0 = time.time()
cmd = f'{LMPLZ} -o {ORDER} --discount_fallback < {CORPUS_PATH} > {ARPA_PATH} 2> {LMPLZ_LOG}'
result = os.system(cmd)
print(f'lmplz done ({int(time.time()-t0)}s, exit={result})')

if not os.path.exists(ARPA_PATH) or os.path.getsize(ARPA_PATH) < 1024:
    if os.path.exists(LMPLZ_LOG):
        with open(LMPLZ_LOG, 'r', encoding='utf-8', errors='ignore') as f:
            print('--- lmplz stderr (ilk 50 satir) ---')
            for i, line in enumerate(f):
                if i >= 50: break
                print(line.rstrip())
    raise RuntimeError(f'ARPA dosyasi olusmadi veya cok kucuk: {ARPA_PATH}')

with open(ARPA_PATH, 'r', encoding='utf-8', errors='ignore') as f:
    first_line = f.readline().strip()
if not first_line.startswith('\\data\\'):
    print('--- ARPA basligi (ilk 5 satir) ---')
    with open(ARPA_PATH, 'r', encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f):
            if i >= 5: break
            print(repr(line))
    raise RuntimeError(f'ARPA basligi bozuk: {first_line!r} (beklenen \\\\data\\\\). Stderr ARPA\'ya karismis olabilir.')

arpa_size = os.path.getsize(ARPA_PATH) / 1e6
print(f'ARPA dosya: {ARPA_PATH} ({arpa_size:.1f} MB) basligi OK')

print(f'\nbuild_binary cevrim...')
t0 = time.time()
cmd = f'{BUILD_BINARY} {ARPA_PATH} {KLM_PATH}'
result = os.system(cmd)

if result != 0 or not os.path.exists(KLM_PATH):
    raise RuntimeError(f'build_binary basarisiz (exit={result}). KLM dosyasi yok: {KLM_PATH}')

klm_size = os.path.getsize(KLM_PATH) / 1e6
print(f'build_binary done ({int(time.time()-t0)}s, exit={result})')
print(f'Binary KenLM: {KLM_PATH} ({klm_size:.1f} MB)')

## 4) Unigrams türet (filter: len≥2, freq≥3)

In [ ]:
UNIGRAMS_PATH = f'{WORK_DIR}/unigrams_konusma.txt'

print('Unigrams turetiliyor...')
counter = Counter()
with open(CORPUS_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        for w in line.strip().lower().split():
            if len(w) >= 2:
                counter[w] += 1

unigrams = [w for w, c in counter.items() if c >= 3]
with open(UNIGRAMS_PATH, 'w', encoding='utf-8') as f:
    f.write('\n'.join(unigrams))

print(f'Toplam unique kelime: {len(counter):,}')
print(f'Filter sonrasi (len>=2, freq>=3): {len(unigrams):,}')
print(f'Top 20 frekans:')
for w, c in counter.most_common(20):
    print(f'  {w}: {c:,}')

## 5) Test — KenLM perplexity

In [ ]:
model = kenlm.Model(KLM_PATH)
print(f'KenLM yuklendi (order={model.order})')

test_sentences = [
    'biraz daha bakinca aslinda su an',
    'islere layik degilim',
    'merhaba dunya nasilsin',
    'aciklarinda bir ada berna',  # eski Wikipedia KenLM'in tercihi
    'bilimler donemler',  # eski KenLM'in tercihi
    'gokcenin astroloji kanali',
]

print(f'\nPerplexity test (dusuk = daha olasi cumle):')
for s in test_sentences:
    perp = model.perplexity(s)
    score = model.score(s)
    print(f'  perp={perp:8.1f} score={score:7.2f} | "{s}"')

print('\n--- Beklenti ---')
print('Konusma KenLM "biraz daha bakinca aslinda"yi DUSUK perplexity vermeli (yaygin)')
print('"aciklarinda bir ada berna"yi YUKSEK perplexity vermeli (Wikipedia jargonu)')

## 6) Dataset upload

In [ ]:
OUT_DIR = f'{WORK_DIR}/upload_kenlm_konusma'
os.makedirs(OUT_DIR, exist_ok=True)

# Dosyalari kopyala
shutil.copy(KLM_PATH, f'{OUT_DIR}/lm.klm')          # backend pattern: lm.klm
shutil.copy(UNIGRAMS_PATH, f'{OUT_DIR}/unigrams.txt')
shutil.copy(CORPUS_PATH, f'{OUT_DIR}/corpus.txt')   # backup ve unigrams türetimi için
shutil.copy(ARPA_PATH, f'{OUT_DIR}/lm.arpa')        # text format yedek

# Credentials
KAGGLE_USER = os.environ.get('KAGGLE_USERNAME', 'UNKNOWN')
try:
    from kaggle_secrets import UserSecretsClient
    _us = UserSecretsClient()
    KAGGLE_USER = _us.get_secret('KAGGLE_USERNAME')
    os.environ['KAGGLE_USERNAME'] = KAGGLE_USER
    os.environ['KAGGLE_KEY'] = _us.get_secret('KAGGLE_KEY')
    print(f'Kaggle user: {KAGGLE_USER}')
except Exception as e:
    print(f'Secrets okunamadi: {e}')

DATASET_SLUG = 'lippy-kenlm-konusma'
DATASET_ID = f'{KAGGLE_USER}/{DATASET_SLUG}'
with open(f'{OUT_DIR}/dataset-metadata.json', 'w') as f:
    json.dump({
        'title': 'LipPY KenLM Konusma Dili (lippy-test 43K transcript, domain-matched)',
        'id': DATASET_ID,
        'licenses': [{'name': 'CC0-1.0'}]
    }, f)

print(f'\nUpload target: {DATASET_ID}')
print(f'Dosyalar:')
for f in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(f'{OUT_DIR}/{f}') / 1e6
    print(f'  {f}: {sz:.1f} MB')

if KAGGLE_USER != 'UNKNOWN':
    print(f'\nUploading...')
    r = subprocess.run(['kaggle', 'datasets', 'create', '-p', OUT_DIR, '--dir-mode', 'zip'],
                       capture_output=True, text=True, timeout=600)
    if r.returncode == 0:
        print('CREATE OK')
    else:
        print(f'CREATE fail (rc={r.returncode}), VERSION deneniyor...')
        rv = subprocess.run(['kaggle', 'datasets', 'version', '-p', OUT_DIR,
                             '-m', 'konusma KenLM v1', '--dir-mode', 'zip'],
                            capture_output=True, text=True, timeout=600)
        print(f'VERSION rc={rv.returncode}')
        if rv.stderr: print(f'  stderr: {rv.stderr[-300:]}')
else:
    print('\nUpload SKIP (secrets yok)')
    print(f'Manuel indir: {OUT_DIR}/')

print(f'\n{"=" * 70}')
print('SONRAKI ADIM:')
print(f'{"=" * 70}')
print('1. Bu dataset\'i lokal\'e indir')
print('   kaggle datasets download {DATASET_ID} -p checkpoints/v1c-konusma')
print('2. backend/inference.py\'da kenlm_candidates listesine ekle:')
print('   "checkpoints/v1c-konusma/lm.klm"  (en üste)')
print('3. default_decode="beam_lm" yap')
print('4. Backend restart, ayni video tekrar test')
print('5. Beklenti: konusma KenLM ile WER duser, kelimeler dogru cikar')
print('=' * 70)